# CDCN CVPR 2020 · Lượt chạy thử Train/Dev trên OULU-NPU
Chọn **GPU A100** và chạy các cell theo thứ tự. Lượt này dùng 3 video Train và
3 video Dev, mỗi video có 8 khung. Chạy 3 epoch từ khởi tạo mới. Mỗi epoch lấy
ngẫu nhiên một khung mỗi video Train. Dev dùng đủ 8 khung và tính trung bình
các tỷ số tổng depth dự đoán trên tổng map PRNet.

Cell 3 khôi phục ZIP tiền xử lý trên Drive nếu dấu vân tay cấu hình trùng.
Nếu cần dựng từ đầu, cell đó đọc các video được chọn trong TAR và ghi
snapshot lên Drive. Cell 4 lưu checkpoint, điểm video, chỉ số Dev và thời
gian. Chỉ số của 3 video Dev dùng để kiểm tra pipeline. Mốc của paper dùng
toàn bộ Protocol 1 và chấm trên Test.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-train-pilot")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-train-pilot-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn runtime GPU trước khi tiếp tục")
print("GPU:", torch.cuda.get_device_name(), "| PyTorch:", torch.__version__)

In [ ]:
#@title 2. Nạp code đi kèm
BUNDLE_SHA256 = 'd24fa15655a59216a83b749f133ad7c51593c63227a3adc7d7ae0abb4a1e52f6'
BUNDLE = (
    'H4sIAAAAAAAC/+19a3PbyLFoPqtq/wMOUnUD2CTEhyjJqmXqaGXZ8Y0fupa8ubkKCwsCQwkrEEAAUJZWR+e3n37MAIMHKdnrbCU3'
    'YtkiOI+emZ6enu6enkae+duBEOnC84WbesG264ZxWLiuk97+7ht9BvDZ3dmhb/g0vweDvb3qGdOH49F4+Dtj8Lvf4LPKCy+D5n/3'
    '7/kxTfMlTP8rmH7j5PClkYlceJl/aaSef+VdCAcKfLf13ZbrXossD5PYdY2pYQ6coTOAjN89ff61P3lz/UdJnov8G67+B9f/3t5k'
    '3Fz/g+Hoaf3/Fp9FliwN112silUmYG2HyzTJCsOL46TwCljvOa5+mVokwBm+26I69KyKx7Ge6sSxylisYh+heJHh5cYrhPXdlh95'
    'eW4cJXGReXkRXouXIi0u3wLhWXHsvEuCVSTsg++2DPgEYmGoLcnKRbSwjf4fjfdJLGQB/OSrVGSW7ZQF7SrvSmSxiHJgWuezKvX3'
    'xnF4cVkYqzj8+0oYizDLi36SBSIzgnCxEJmIfZEbXpas4sAoLoXhC+iuMNLwRkSOYZxdCh1amonrMFnlOO5ILKEsIc8IYz9aBSIA'
    'qEBnALNffE6MZLHIRQGFY8MzxjdjHRL3FyYgMABPy1RQ80tjDvwY0GvAyHtGHkbQRHRrBKs0Cn1oK74wFskq0yEFYSYI+blTJS8S'
    'GOFtzwhusPXzKgM/Vn/YM/pDu9eRPOhMbZcddEIYdBXtbqyzrWZTs4N6AYm1qSTAX0SW5Na4Z4ztrnLnCG4GpYf1XMDLDaYazwlH'
    '9H3TCQBLIoD+sCs7d7w0FXFg8U+tD0jBTiYugBpE5s5XSGqWKWuZPdl9IBX/SlbO7fODHhH8zMbFoxYFTORnLwtoTfSQ/oKQJvtA'
    'wjgTcZ5kANHLLkRRT6U1pCccdK0X6qsaUJFYAZC4L6ZVUw6nADEVt2k9AxO0YXMncHro4VdCwyz3IvMCAPjK8ZP4ehRYVfmeGgOg'
    'xQsCWBrTYasvrfqc/EDdTACbjKFSNHRxo7TKrvR0uLbO5z4sFqEfetHj+B2IW1pBmGkoaWBTBvNX4ESJBGgcvTx6bxz9ePLRGA1G'
    'AwMqAfsDRuAnQSm3fWMeykRTENFYddJv8BJKwoUGS38w6xnnxBiqZ0qe9boqIbP4mkpY72sqUcHh11SSLQ2/ppJq9csrKfx8aSXC'
    'T7NS8zevPp7nRZR4xXiklfgCViafnrjXl3EvR9yksPnT7rrP84ebZMWXLkAmSfPp/mNZ2tcALPncMhePZnSvEt+LHinF9QwvSi+9'
    'A4NIDDo9cEYT6Iq3XGqJI2fwpayKpp1g9/iZQAIsmUY/11NilFyERd5NhfnjyVBWKEkm56Wkd3QOCjfO1TyMvezW9UFgyV2UMZP0'
    '1v0cFpcu98Xir7ITPZicYEWkMjVjQIxZo61k7s3DKCxuS2adhxfLJAwkHL1wUZb5fAkSr1X2eor8RYOFolBfT9CgEF7dzaD0SUFI'
    '1e82zVkK4DOQ+rDVwnbS5LNVTaYNWYA921kKL7YkCeJU0mZZkevjOQrMu7ecB57rzfOK+obOoMzw5Z6sE+xkHQV8O34hcVL1DoZe'
    'Fz6UyMNN2SCxNnoMNTolD7ureoVLJWS4fuDHroZZ7tha/OoY6Mxah+labhvdXGQdxkHY+VCTirCrB1J6end6bKQR6GYCVb5+jF9z'
    '0JWMEkVQQpeX/tHzV2Oqj5lATdLaKE2umVO5cT/O/gOr/CITef7b2X8nk2HL/jMe7z7Zf34j+++JyMIEqMZQU09GgjgpxDxJroyf'
    'EyBbNEdESZLmuE62HjYabXGZ1Csuo3CuCpzAT2VLyldzaM+H5krr0mUmPFS5yoRwKba2pHRxIjt3sE6giLy5iFCDLrxo+p7MJCvI'
    'n5phIZYokIYxCKrXkDke2Ad1gUHWpWcCIJ8RgHxUtYEt6C1xIxXwOtwAuoF7RbO1vHBXaeAVQgmqSISFQOkNB+0sE8BkEoe+Zder'
    '5kWSYiGFKuf4GmQGKFViheFKnGD72ljDBaUY/zGtundQk/71XuNXO7Pe+VZvy34sYa68C9URGF+h9yROPm8aq4i8NCd0YMF+DUdl'
    'IdnNsstlhp+sYmTfC9O4oyH27rfvqrmFn8ZdOb33JqKlyjXCHGmfZE7oRy5gc6n6VaAoqSVg1YJQAcnZKkb926TVokHEnwPje+7w'
    '91pOHfdhEIn6iDVc10pyNywY338Zx2eHxn/fWVpzfZ522DcUGrcp5cAZLO5zGi419X2DsJ8Zo5Y6SXMBOKCm4sSIxWeJXuQRdwhH'
    'grWbgtzCPL+rFtf9DHCOmLq/o/r3AE91704+MKQ7GN29WRGSeylg3ucCxGeyXVRI+3wZwjBwssq14Xz2Qi5XjsuuYznNIINLKBIt'
    'Jw60ogUICpfTs2wlNFp2UTAHJddt9qALmCTTJrDaIuL1W1vIZ/QkBecplapGDivZE7BQ1sPi1WG1ZgGL1AZyo7FMceO7KK3wE+Bq'
    'BY8gVvgCrc1NJkn4zUVhdXfg5ySMtawu1Cw8mLCAKFA1LZfYAlh5ftnG2hYJozA/qA1diMDyE9AAYtBCn5UcH3J6hv85kExfxNf0'
    'JLsP+9UrVvGAcJegbQHB+UA4gZGsinSFhxrSvh/xSk3xCDS7Fmxry8KLEE8vRJYlGW1+JKYmFzB5uJ2hWmWrNCf1MqAUZ3kVhJnF'
    'P3IaCWI4hMWcXGlTKMeCRxN5kVkw8zYtK3hA47zMnm1V6FyYb5OLA+MO2ro32/SFaiP1I0lFbJk3JmLDT8iMaa6KRX/ftPEcBkbe'
    'KzdUS6Kx3B539e2RQFZ7tXNCoMtZQLTDf8Y6/EdOHwD4qV7lzclxr4uz6FtLEQCG9VqnZy8/fDoDchQ3hUJhYyw9npV8amYijYBq'
    'eXQIob7mi+z2oNUBxHQE5ICoxioO9/ygs6eQ4XzOQJKwsIq9tgzNiNWdz1OI9XEowdQ0uzlEtY0FuB9Q34it1YvACgLB3/jBy8Ux'
    'PaIy1gICS40ApEkUWTZubu9bm75mPfAdIIIlUDyMtN2jTjyWNamPuKXj/E/atWWHtTk+48LHN2kIessG0Fch9v6Bptv5mRfmQt+r'
    'EaX1ZqiI8RF2JOjLMZITLLI73rJwzaJQRmuAZuMO/947xqsVMAvmH2o12v+avhAt/W8JI4zyb+oG9JD/z2Sn6f+zMxk+6X+/3fm/'
    'w/adS5RJpO5FRoU/QYI82Hc0O5BerrRHvDx6/xdYJ931yiMrVevVqzdHbw7fuljNPfn44cfj94fvj457NXgKxDKZ43m3KNw5MDvm'
    '2QznHeW8F8UPMoNdlbwoIjcleSRllp0ypZnJbFUsc/QOdCbWRlmVWDMiLDD7J/aTWrf+a9P2K5nA5vU/HE12J431P5mMJk/r/zey'
    '/5DBFH3+YLPz0aJj4NqCfa88cfZWxSXIWX/QDptLP0E6bMZld3YJwsWSjn1KGTqn6pF3KzKQs0EGSS5ujWRh/PQTQkJAQHMpaFYI'
    'JVe0hx1C++NPP323xTqeF3gp+bggOKnZkLqipHeQAn6Gzv8hN6T508tu+8gtAoOAGocnb5TXDqjTkAESMOicqyDps5X4AhhM5hV0'
    'RKAO0nMfVBIYT48UA4EtYgnowRJKQMNJjsfx320VOHYQQpM8hPxbkIITVFoAH0BexhzP8XM/C+fQHQ+NaXV0Q8UskWc6xioGSfi7'
    'LbRPQ2qR+EkEQgqMETB4iTXrxQGZNH7EIow+Xc0jUqSMeLWciwxdf8iy/UV+Xkto79u7fa1jkcCo7yQXrXtPmQeG+f8AofHFX1fb'
    'NX5coRrLXBZFmh9sb18AZ17NHVBOtrurQQEXVZewwGqLYH+8NxD7i/FoHgSDFwNvDHxnMfGGC7HjjwfDhffixWI0L6vnMCnAJRdA'
    'kVj/MSTcrJtfesDu3GiBAMYvducTf/BiPN+HVhcvvAAEn53RZG8+2R+LvYkPGTu7vj8JvOF470WwvzN8sSd29/0X+6N9b2dSAtep'
    'iXqmD5p6h6mntGCRro/wnBHm5mXp6oZnRNdJtJKTBhvj5yS7Yksw+QYfgoDcP02TZIEArJIR2FUncI0yKUFrmjeIWS1JpXPDJKRQ'
    'dA5Lm4RrtQP102pL5aq0Nn0gfHTvgpXvkQIPRPXDzfBmPIJ/oL/lBUoksBTyv6+E+IUYRQIlKL8Gb5F5SwHrOoFlFzJ3yrEnMGY8'
    'TYSl6nurXJRLrx/CckRXLsCYcfIR8AKLI79Cz0ABS9W79kLQFSJRa+To8PTNIeo5yJL6QZijaQQrxuFC5AUxk/HLl68OjR9HdSYk'
    '9djakv7w6e2nbW46DdOasCKHlPwiYoPQR06KoW9wD2iw3FXgx+EvUpWBOQAuQpa8LEkRU6XJ43iAz1I9Y1cinig69VXNosvIfaeL'
    'E9NVRVZH7IjQ6eXEZfuVuyWexykaRBbEtFrbf2DfIIiuH/z001oXp7o3gIarMHYlKeUHaO/QsgApa/PYU8JFDFIWMKyxlp0DyoMy'
    'Z6jlSBeyrqwA6IaPT9t57JDRlTMPPUifJwkeQ7zyolxomYCpouZRsSczv8qHAicDrdGxIyex7iakobLhQKRjstflHUmInGrPjVKM'
    'zyl/NfKUU578bvouSaRO1UMjX3q68FcjD1E7xT8bvJ0Iw2S1he/1TiTh0rsQj3cWiZNsSedKJd4tgqC1Hi5oT3YW3jy3qq7YxvfG'
    'UPT3G86wUkBiuLpH8Ilu2gQUFJnwlpU/icEO0OLG84vo1jEOjWIF+7FWwPdiHZ7n+6vlKsLzD6I6mJF+SryOhDPp4KxWeCHhI/sD'
    'pAmvIAasAwShRQD/ArYg/r4KAQjzhUvhX4H4BVzu86WI2TCbsrSGXBxRAxJZgRWjWx1eiHwbk53WasZe6Th3PpN7gJOvltbIll/o'
    'vCb916QXm76CS8al+Vw1VgnOY/ciwOpdJFh1qEGM+tIoy2xcJINu8q9qtxZC2x1HEmdfXwDPtLFXHiOltujOo8S/sipuSlR0jjzV'
    'qP2Z9Wo8i9YI8JtTmHucNi+SdE3e+XgMC51G/7PLMIMvdHeHLXOqpIO85nJRg9PlP7Fmq2q1Bf2b8mLT8ATQf8AN6j1gZ4TLHks3'
    'CnwUbz9ZetpDTTfG97imqfCvbbmG0ce2zKUfahq3fu/mBHaskXKHZ/4/7ilqHum+1r3SXaVDvACxtlOWOKvpqKjOlsomiZee0g/7'
    'muBZiqKIE7/Y6DDd69hcv3pXHfK22kmej5ktwNzuztpZ6pip3Z2OAq2ZanaVVjH2tbmwLWx9ONqHPy926cmWvekEMeoCwdUfDWP8'
    '62DgOfq3wDw0AcxvLFt9/ARQ1758BlS3R7++278NyagOj39lh4nAHurug71hncctEjZbc58+pbmHRgaLmJA1BuYzHgHdoN4+NVHP'
    'iUFNNu0vl+5Qy8Vz3F4tS7+oRHuLkjl4eVkVV5BCX20UuCHUKox4g9J9v5F318qM1X6ktbxibxrume8VjQmx2hcJulAoG+89snR7'
    'W9xYvLWV0aS2RPzldNg96WqW66Ro1ZdS/efQIszYdlvwuau3y5YIU/qUNjplkkFBZbJfMnYUr60hgdn6KO7/qU4DNtr/q+OeX3UC'
    '8MD533Bn1PT/3B082f//We7/btXMwJ1WYOntuekYceNpoCbflf6ea4/cdAnwa6S1DcJaQ/Lw/Ks5+xjWJFDekOplL9mdq+qk5j1G'
    'xwmyY7i6SgMOuX20+8PlsaStGwDQz82Lb9GtyFuKQmTAQEFJBi7KF3/QVFvmoeZdG4RTZuWWbXc4fZYF8ZjjIT+yX7cjdnNs1QG5'
    'D55LjjsrS9MFFlUaEW5RiVZX7xq82jCpJvym717FrjsuyNj3/14xLdbx/2/E+h/l/7G3N2ye/w52xk/8/1+I/0um/QVM+isYMp0j'
    '5y3dojJR8w3Hcc2eUKoKYUxnK+xs19MM26hA7n5BtUM6jL4Wh9cX0q4xtBs9JWyEixAYMfX2LWkVFjYztNexUVpyD5mNu/iy1pyl'
    'IUqyR2cReUUhYuxl+Vx6q61b/22fm6/nAw/Jf8O9UVP+m+w9xX/5l43/ch1ioCiHKanlquX+OHZPl+ii9ReyuOcoFSliux67Oebp'
    '5r+Wr9aj7hensEAyOi2uy1tfaLXjUwFkOusH4Lw8fnX46e0ZO9uqZtmzHZtq2CXYIWXaMWpLtjaV33ZXTW21n/cpiknFYNaWcsLY'
    'xRMXmOQcedA/wrxRil9K3Kp6I2W6iv3cP4UO+yeR//wsTAup76cZUmO+TK7EtwwAtpn/j2EDGLTufz75//5m/n8f0UwHK3vlF3R4'
    '0s+L20gY786O3r+XjicgBKHTSE5Rq+iKCvBU9FQLDKIWWOrFo2+GqrTsAnTSXKjffn6tHi+9HK+Nqp/81ZXggEbroT+fyvk5T2Ld'
    'i009J/naC6k94wT6epLk4U3tfurlqgij9bdV89vq4qqXoU9Y+VMs09pvvMaqnm9TUaHAvx6px3i1TG/RrylOuae5H6a3jnJOkqX+'
    'z+UqiuiWgjS56KKbU97fVaOTv7e2YBaPz9yjD+/evTnDC4wLfzhaiIkYLoY7u6N5ILzJaGfwYj4ajoOBvxj6+/5i15/Mza2Xxydn'
    'f9Jq+rvBcLAYifmuN3+xO/Lm4+HOYn8ovB3PGwy8xTB4MQ7mc6j5l+M3r/905p7+6RDrTV4E4/H+fDHwR8P5cDAX+8PxYri32N8Z'
    '7Ix8yPQm88V4uC8m+/v7k8liMZqP/MHO0B9N5vMXQ8/cevP+5fH/VeD8+f7+8MVo4S/2dnbm+2IE3RLerjfa3R/6wYvBYjgYeaPg'
    'xd6eNx/OPX88GO3Og8WOL/b2J1DF3Dp99+HPx+7J4dGfD18fn+J1Sj5N5NhGiyj5PJ2OnOGuMzJ7MmfRvxKZl8v0gZaeR+FyOh1S'
    'UEyZuiz8OMa0QZVGk4xpo11nRyXCRF+FRZ92yOl04Ix2qgpEBAR4zxmqRLyB5V/309viMon7qKJHMMnT6Y4zxNb2d6GgLcd3+vbD'
    'GQ1uQArOnr118gaSynQ6mrcyL74Q1r6t7tvll56FK0XqbnSGir/lvbJszhet2H+kpZLI1evgGnCD8ELkhcVF0fRCDpCm7VyKG5mn'
    'Wo0SL3BZaLRoJ8JQJ+gNKzLZkSR3RHwdZkl8bp69cj+dHrtvj18fHv3V/fPxx8NTE4Uhc2i2yh59enno/vjm9M0Pb4/dl8c/vjk6'
    'loW5bLlw1dQ7fE7sXA9xnMWC3RBwwdG0ljItskhW535vnL0ykC7+kNed5mhY5FWTCp89XowkClR8Vcj2iTE6BAa4isM+1Pk5EJZL'
    'BOdE4sLzb134zZod9b1igvzkcj2rqga6Xbuq/QXtOOz9Ef7C7ribWq38ETc234DIZ/1SEkUH1gQVZkKrCrlhHp18OhiobqNbYhKj'
    'wTNFw2GxcF7jo6Xdf6yVcbzcBdryVhHQ2YHmGfQWZoXdLMMYhXUZOAPalx4DZ6+G/TDun70aNeaTx+1oTn5kwySKNbYNs4SE7r9b'
    'zct8uIrwuqyL9xqt1u1MR3qAWvXzpRaBMln+Le6iXHQBnStjCZbzo7/FrJ21i5GPKPr6GtW27OY+LHSz90Afmouk6s3CRYbYaFsm'
    'bmimGj1ySpcdNPJtYMBGLQFS7Fo4rVASDu2uUjU8g2ermgytRo32qwIzmkT5q1aYJi2Mc5EVyEfxri7Ntmb1ad2MFDfCx2uyKTBB'
    'Cydf1kOuCsPBbLzrXBGL6+KD69ZvMRLqvDQst/SP72v5SpeEdCsEUoeFOWW/UAS9CG+mXb1d4JXmqNHjcqRpkloDu5UnUUbZGtrY'
    'L86WvlfIrjsngu3JtMnJqZDrRrF4XDoocSnTMxUtl1ANndUimt8WIrfsbtwy5AZi5R6luA2fyERlYXkLRLhaJ9Ql9EpMtmjV9NZB'
    'kYwmu5ijd9r1yPGvi6MkwqulXA/Tjj68/fDR/eH1x9HH1z9w31CQQ0uDAuvwg0vpFoDjYvIsiFK1zZcusv6Ifpd8jdUkNgp6e0DF'
    'GRYwtgVdlKZ6Nz0Mefq5Z+BELL0bi2BiaLjbKUchMjB4ygH9PTfnyY05Ox/NjGe1hPHMlo88r3gzAyV9MnGgKUJKzjiCHsi3jpdl'
    '3q11foPRVp9j89CJW3i6nEkavYaVFjIuGALGm1OJVtmA2jhSjZAsVaynVY28OFh62ZVet4fz44A4guEPwvydCF6FUSEy7TK2ikhW'
    'ocmWVlpJEug/75IilFuAgJ4hweW+FwmQ2saSEmp4hoKUeEk2HkgOAz4Ew7rnB6OZlCf+DKI9X58iAQLkCvZ5rRx1vaLpdkvRiVk1'
    '87IQliDHGeDoFdAL+YTB4SzEPf35bNuw+EY9w8JJoD+XnMT99KKFi33Hb+oojg5o4DPXUj8vyxqRWBSSoIBdsuNylABJqo4YfQlV'
    'zndGsZihRhhbhI7uWs/rtTgUztpWblUrl7L8PCkK4KbcjMJ+V73n9Xqw3mhEf5zKjiZ0mQ1/M8hNixB0QZAZCEcBLT2cf7NGX7At'
    'wpYHEHsSHvIzSsNme9xoKZqLCFnCdRiIBMVkvqYCFdIoZGISLns48T6gi/BST2UpvqpqZgcsz9P1oGuhDwej9MhkBwQWdJREENbC'
    'PJH182315A6376gb905xU5AsgwfhwMVw/7NYxOnn4YUafvKZwnJbFBYBb9zj+QowSPJqTi2bo2HI8BSYQVEx8msCDEsduudQg2gU'
    'xmNtmiosriCotfQhFgahjCAKD99vAMqT2JbCogwNwouL7vphEca1oEDcuVxLVM2V1bj7HJvasHZ6xsieYR+0WWArsCzE3fEvE8nc'
    'zi0C11Myaw896uGRB17PozAV9cbJEEMVMFM6G+3IVphC8G7d/ZYKd9HdGmGV+3RQixPewDzOV11kWIQiCmggOIW4J3DP8Qlr8GTi'
    'DFmma9qzrUZkCosBnGOAWvk4rB5hX8GwSutw1BnrgumIwk7aGObKkkOkhO4AE601uzAZyDLM6RbSgXGHA7k32wEmGMnnmI3iIzfW'
    'KjUHer3a0uMpdcWgqPUA2qb7V9K6BguvWME+LNeXcUc4oTBWhBR44rbv66yF+ydZh1iGknGQqKQQWfIOFq/lvonJUVLkU82OIBHo'
    'e6mUa35EYEdeisY+qxTBJHfWJWIVkQvJBKrjpmyREHR4ghdBT9xXHw/fHbtHHz69P7PtesQOrPm9MRw8iDOfrIwk6GAIE8H33xgC'
    'oI6kxuZEIk33DOkdeT5DKtMDifzeeIcUwIoiXq31jZfiutqR42RJ0YHYk90xXlGLb17ypTuYIr6D7NSWFeKV/HMQv/VRKXgcg54K'
    'PjMsxsH2NiChoR0ANBqkC0IerlCy40ggPdrnFMTnxgR2SYb0HNZC9/LRgSFSulcM6o5hvBLtiDUeB4iqTe3Jh1Oe3tNeBb+P5+Ct'
    'ADpXsgSSF4Di/aOrnzjPyRVuxHghqpOiqmZtaE01bOOuPXAmXziwtYFvSOqrJNCWhvCMF1N36JosCXGkTSGSqrJkujaIjlVRfk+z'
    'CttfOK5QBs3l1pBhao2jLIoo5hIU1hRLgAC/gnW8/0huarLhCy/tGgvoZz8HOUsYBEJibumlZjc1kH1EBO6yZDnhktMs0/k5vTAl'
    '8jcQyaZufjiVfYR1EbGaxNBlx/73yfHrjp5Bb8Yj2R92Yra4a1LS0XrNut6bdx+PD1+6rz8e/vX06PDtsX0OUw/MRrmfr+0+teSg'
    '15+NM6HvZiSmYBmYj7RILSpqfykB6AxOHu9ovE6aLjOBD7kxF4tE3SXGkHVtt27c9zHuIjNb/Hnv3qm1dzAYB/dmuxLSgyutAfJS'
    'zjbvSWgTIBU8N+GxhAnP0ARDp9pECl3TtB4sZK4HSrM/fNkNturvF4Z76+rbrwAhSYQpj2OT4RZcdc+WrMR+zBJYaGuAgIFsUUK6'
    'NzeMgEpLS4xG+U6RSOuMvQmFFOk8Xy0W4Q0s6cBDvYEhkoHU/FtscmRBAGmR+wHwSBvDJGNWGzTuWo4XBFbJ8dtRwWDPVy9ouTNJ'
    'IIJy5kFFC3hOgYSCPgwsG5m88iBBCU0m7s5YAL4eiG5nqr5AcfWIJk3emiFR7dgPgJHH86ulOx6ZdD2cVz1dVrXR9EWuJ64M0I8v'
    'wSk7jJx7eG8/IJtKsRExtNVpKORtORIg1ZTnNhgLRRnX6UQ3I6WRT3edw+xihRE9TijH4kAoFKxu6rpB4ivDHNfEqXM9WcUy+308'
    '3MX34mCQbTye3ViYzYuPLY1aWp+1NBOFYQ6pnxfA41zYxDHxUkTp1AQ9OjGkOkeCt7FTqWUjc2MjKpZMPw2RXja2g6EXUG4k04SK'
    '3pACOolGN7dDtKEPHY2R5B4eaPxjTWVlg/3a+sxeH6wNVXI6AyEg9IVgcqu0s2AMzBy5gKciFbIKSMk0u2W6FoSV4QneyvNVmka3'
    'gEMKmcMEhHunoo7SwqgUeE0/vhK3PQrj0zNKqQU1a8tU6KWOyMNJ/VAd9X9TQyOVU797hn6Kru/RHIwGbWpVUES67S7POqxzjFmD'
    '5x/9I5Ot3Ng9XOqZuO7TyDH3TyBemDMtWKatLCB1XYpaA06gRvegTnUHKLnnC/gYNkSFM7pTAPSdQeH0HOqgNszNbVVedMwWJP5w'
    'E34Jc7MNzzRJ2xgYB/+DfLHwY3ySV/Ixt0+uQf1kQQ/yjDfko2ouRvq+5nSAmlaZ2dhnMKrLjWlTldKvYJMZTwpE7A1D6GCDK1Tr'
    'Q0/lhXh1ukqj5BncvLOzia6y0JUxWU06n2QLPgp61LCpUY50rpzWTs116qxRYLnAqoWkRQWndaXEa1K6ZDhaWbSukdeW4eOVco6f'
    'WQHVKWetFsXxE9palJR9WDMh8fiBjpg8c7y/oS4SG2yZreseTWFKm0ikV24v5UjRDcWDo7uazBTevDt8jc4lp6dmj0eHmfJ2oHdD'
    'mzWmgOxOKrkbJzFu19Jle03oaO79VmW3wHjK6NEpbRfnbGAjuwKeCWqeHmrq1Xbk0nbExknNwsP1N8eiJxMGidHIGs0zhIg86KW4'
    '1gm0tD/WLdYVgwd8ljZkB5o0S0MUldFsqBUGpClaifU6MBTeyThG8bkYYmNyyiJ4Eyi5opEc/vjGWAoKlYbjudNbkNai7smor9cS'
    'tHF2+FFGOME7SD0jErHFY0euzU9lnGKq3DP+1jY46KZ6vU9tc32vdDhzzgSeEnvZ7Ut69WOSod6Ip/CQ3lggFMEOLYxTMnnJHras'
    'S4waiuPSPBwobVZssKk50VlcjxC4XrnlMmFOJwqk4LLBmLi0EyWf0SsbWbTpeNehWRVAdS1mI1o5kC9UfNHtJlIxvBE/eOBF0PFP'
    'qzg7hXQchPAg2B2KQrz1GLR0lfosXaVA7F0TWZr9DIENpLcIMJn/bClAWKln7BjPjOFgpL46DCVzDOrjqTNXzcxLPenpVmrC3ay+'
    '0jrtvmvUJhg6qk16k3Y3spq8k8VZBx00I9DxGD3eugjlPwMK4uYLEnTC1HqAZKB36GCtGiWhykje2BUnWC3TXB/PetWSzgHj+tiR'
    'OjGRcbamu5JLd+qc5aT0FIrwnmKzmQ7FjalsFUdhfNUV+lqtC5ABlsm1sMqG2mXLV/7Id4doLMvoU2dKaHZX7QZvvdtU/35bz75X'
    '28R/GXfNzaYv96EDZ4jvl9gcK12ylAe4QV3Zrc2NJOza3NAjOpSgCIERFNWpTLVpmPdEjXy2iDZ9ipzawpd09qONOgfJWR2KTaem'
    'fT6YyT1V+Aii7pg6k8c55P0DykpFJCwCmGoEaFuQjz0VbhLHIIVySGNJWIaghKxKVm4aHlRREpOrCqWk3ENbhDz0LXNR2N6wu9vt'
    'RqodHs0UjU1fWlhoOYToQUorrAVEYdaUp3QHHc7hjkQCrQE+oVRnmqr6fQswB69AuE3CxGBYTJqsOTY5HZ3ZoZlmCTsxMby6TUvj'
    'OzyvKAtioLLpqPI4UHO5QSlZmKd0OhjG6BgViQIGf6fqKfFaiqUy5CrKpoq+SDalxQmEZFfsp/kCjhCvViG+MKQ27MWui7TtuiZ3'
    'ja0/T7d5/v+5/8NGRFIQvs0toM33f0Y7e5NJK/7/ztP9n9/q/g9pcPzOKoqLW93twdCzxsmQQ6qxxkiynaaiC2mbkKdHX30JiOrA'
    'ZhHJV9arStJ5G104a3d8Hnq9HL87bvM9G5mEb8f1yFM6DdbddV8f21pexPFWAQaXDnwXls7QJUyWg3ifxK/waqkAHRnf85RfJlHQ'
    'Y3TLkTeu35VDqnT4nn5STK6GHVeA8LWSVbubXhbZUZkVgIdjtPTWvr/hiy4lVZc+pG0gS5LCVruKjJU8pel26ADSohJrt9gNbvXl'
    'tqrgnldyE51xq3TyI5CCBwv3KHtos2BvshJ+ynH1qN1Yvm+UPRrKmPK8jPLYS4EKCnWvAcVBMkZLe8qBTvgWdsKWJpaOnMqADVs5'
    'SjXaKHlPnzXsMmhmSz6fV8LubKvl00V5mvfWVs3NCuQGLklY2jH4jZEGXmKmdFDrvdwPQzRAAGdRaUF4gW/qYc8VTMPuSkBtxTbj'
    'XpbHZzN0BMJk0jjYS6w/nFEXqKQ8pZutBdU+IJuRm1itNp2V2faDts03MbJBfP85RrwlvL55aSjHORDGKE03dPI8n1NrfMI4m51T'
    'qZnSDiFPRoiQAcQbZxRSc5d6E6ocBNOht1zqdtDyBGNq7NZJXxe8Z2wDHG81tFylv2mHBTiyOxQWQWPRkCXpjvrCtRyO/muBegfV'
    '76Rf4v2j8CmXjrKm8WqRBrrLULfVKvScU1E8cLhDsmiS7floZtd1NO7kfc0nS6p6jXG0cFraQVDPszQ3eUg8YAqjY+GZ3XJ1PNdy'
    '9bWK4GadnKb7JWMZTc9dc2E0IN6XlgkdYoeBYvM0oPfPClDpU4hnPpCkFz/WKBuPYdRkSP41M/6XloZ8a7aJcVYiCLrXwdYGYoiq'
    '3nCWJmrXNw/qFu0MNH/SIV0OVc4r26tbTK9n6CtRvdh6Ed6wEwsP0iVc/0Hh+g8zzY+FfZkQbrLGaYWpjszTDLrlslKeezRgKA+V'
    'Dggt/5Q1RzjUkvLHVC+1bpThE4c1fkpbXWc+5P/O0LQUvq9TOY79R/n6bErbqMe+k3YV3Ihh34jCCzyj+fj6h20ZE0eN3FSe/Dfl'
    '3VX0C6FbED2ejU6Pkko0sOse48oTvdMRTxuSoqoWtqX/FyUpr67RZBdY3mS3C/OyPKd1uoHhaqIrFejl5YxpVSBjU9gkX49yx6v5'
    'hMw2Ylm9viKFpbWkqNkXGBylgVu5zuTJGzeq7h7QluGWZ0iNlQdCaRZfqMtP9IJf2i2w85BBL2S9wLhEpdHBli7hcrxGrdQQkDKc'
    'oFl+OKjQXp0Tblr80u3/lP1/I+SNOFq+Dtn0b8OJlq+dUS/7e79antwSp6M3+7AuXJt7PLmLwpQn3gHZGm+5QSKRIsyl8bw+mv7O'
    'APZA2Ao5+DUsuAHSyMRuQuW7KJCDnr+cvk0FjWfPCOQKtvEkW1oDZwIIcia23dG8oiSsAEMPEiAaEJ5qnq84PLdyG64jn6KW1rcL'
    'YMpez6AbSHKuqq4MoAL8xV4i5T97Nuq1Ozuqb2nNC1AZLj3LwmYACrVjP3sGNfGulpa3Xcurv+gTL9HI60e18Ywme3gdiB0Fel15'
    '1Au7Hb3/HGAewP/nqrsI/gD/PKcqM7pd7+y8GO4gCnb2RyP63tmdPGoWGlTNbeJrBw76eBeC0svfVOsW7821xrcrSUsiGW8wtVZU'
    'z5gMbHQkH1VUdy4vUN32sYp9cPscv3vqXtWNTL6h5Fn5MvOSqQHRoRwNyhnSjAqn3EWKNREbmVuFBMkOBzO6/YiYIJp3BlutVxHg'
    'a2zc7GIuQwqpcESoeLqk3EtGqbzTOmvpFzStCuUSy13LuW8MR3sOs6PRvt6tVvt4wzGng8eLVbLK+bYjXjYExhnnaZILa0TLf2iX'
    'ERdYuiC6lnHbcOrZLxAaUEdL8piDm8SDUzQqqDhvHr3hVSpirSIarKpkLd34HibXbrpTtIU1EtH8JMMdW4UgNbgddZ+ZLDj0MmW6'
    'b3ktpJEINp6GPMc+EVWIOtrfKMgzolrvHmDqP+WoYhkunPxeLJs3JzSmgGR0bam7uLQpsCTfM+bo5ow4xaACyshAF0S1eKfSajNV'
    'nhQNn5xTMo+hMPuYY/06ow3oDpOl6Rk2sl5BbzzBQzd5clTXPOxe69qSlMKhj90bIN46bekXbbaWl9HA88Lzr6zz1jqRV7mWPd4k'
    'uN1Zk98WXlSB4tgA1nnl7CKlFdpqFJOrgEEKeQzKJaQzjIr8cElUU6LjlO4TI0khyF/UZpxL5Y9mnPgD9rGW2KEHKf7L4cpotSYy'
    'BIVdxaJ1/HTV+Z5h7KTS4LvWMvaycXbKxNZ5KKwOHsmOID1924o33Wnb6DPcCHSLcd+5WzbHKLeJzqzm8XLzPDjQLvjTok0DB133'
    '6LaTxeNQjIyDFzM90mvGGYc9Q6SJf1ktxFyIoGJqnKmsO1QEf2y8WMtVkMlweeWbqFiO5DModOeiAInujN5TD9tgp8PdjyILF7fK'
    '4KC/qg35HVu3df+m0mTXsmC2DQu6nUbZRg7a5NxtROl+c7m2urvPutdzB3k9Dw1hlZMWcx9pn1JKvJx08hKCXY1FGEfaH11It2ga'
    '5W1sJDFA3MqLXEzW8lSUinYY7SmG5taW2pZy2kFztV6jw4JttSomaREugYVlJUeiFIzYurRkLAItDDZw8Ww6FH0Q3OSBdyB873Y6'
    'Ef2J2hD4BaBNgFHmllnOaSHStx+tsnH00xUpv+VmMoB9/sJbLnGkEmjyBV6hF+mKRCIco0M3sPAI1l8FXukBC0U0QYoXOuSjpgma'
    'YSq8K3cplkl26wKnL/IawkC5QS+xir0+wvePFl6lNgzV0qZrhQ0HVRnLXPP0FBdQcw106Q61WETUA5RdU5EtV3xoZOkka2tRwenE'
    'o7ZBNH3z2LzEHb+jr/tt/s7v5V5eA/7gjl4aYxFdFS4GDEuNwZbcrmMJVwtPlj0nUAf09znVahuxeXtTSjnWxk3vmbVJL2+saGls'
    'Zk29yyza4eBTivq6vCD70FqCNVulWhGkIVCUegtpskjQ31WsceWpRMhyO9YbZiTYndtzC5Y3z/GFksopj17UtsyFi0RjVQ0plKIZ'
    'TDKartwOHyy8jFe2AiuAmlnnndSQybH24++kv0/ivhSyqdkmQZvd3aOw+hTeddP8IMOyugFUvlHnlhqoFBuUqyP/sjEujCYWP+xn'
    'xqvneXclFFBy3ndIVuG+9AwPuORUuwctWVglRxE6QOah75502nZh1k35QgiLQIMshetczXdHgeFsg3S12T+I2BxA54FiIavGGGDQ'
    'LLTgPWhUyOnXfdObXPMUksOk465uV7hqQ9JmEy35MGGu7G0H5628mdj+jfsFcZLqdUG4myy9G7WNgPaIMetEoBag3IWqgMcyzEV5'
    'S4IuYKJNu0pz0sLU5IbcuxZyh8ZdSlDMJqSwqkan1PaRrqZqpVp3JOjo2NWAcoOYalXVenSfUcXim5og55tKKsjdJI5uGdsq5NC1'
    '+8BOCUUoVE6nUqqYMp2LNHXTqoVNc6Z1QVUAluj6+bWVqMsK19INwYFUk53MbnTh4vfGIblceCnGP2Dnj0t0jUjwGuViFRkfPxwZ'
    'hXJYYEt0Lp0RMcD1DR4Nhfiu0K3yNvm1Q4CceBWHf1+hX3fd1APCVxb6fNaNs7LC1WGSP4kXF7LHpvaSDl6SFWAQ0H1cvHyAV4+m'
    '0Q7Dxu9sp3mi6giIUG1vNa7Udzpp1KF19x32Mpc5s1viyrzfEO+jAlN2D7cwoAX0gzVLIC4ZlQI69NdKduTP7qV6loNgTt1DesPO'
    'qWuHJkrjFJNb4N1TIjn4KUnPZNrDfDoSIkJsuUTCGq90SWQKeIjOhclBcC0/wJupWKus7brSJdN1W61weFMoipqbXrD+nnNXvlG9'
    'fJ/7Op+Uc/2177NWc7rgJ12vm7IgrqRaASQfmc6WDBdfAM+3NA9ap66tJiUvh6LyiWHBb0kZ5U3eapNpMHLZepWv/Wr7yJY83YVp'
    'qfg3X9rGi89lgZ7Oo+u+vRonbrXA6rGr1OZ6xU1OO9KDVrIsIlaXqfghx1ksoznOSq/9jusEW7qTOtHG2cfDN0AhNEV44PlX4w46'
    'cP+lnq/f/K6zNDL8Yy/hrqnN2pCqTTui1POn440VmZN8eT3iR13VdjbXk5yqp0JWTS1Wh5FygT/ZFSBOf+zVY3ZdJ+CVjl0G2dCY'
    'Xpi73rUXRt4cby2tu4Z8SrfeDM84SqCk8frkk5GtYtzIDXpPF0jwaiwG9buyyrA/ORND44oOd4OrWVqPbVlOGdnoh+Tt9My2mA3+'
    'v53+i9/oJQAPvP9lZzDca/r/jkZP73/5DeP/e5HPr3FHOTyQNgF5GuIV+pE5uQJLYSTBEFhoHS0lkl/7DoDNrr2P9eHl+PlXkfCy'
    '2FHCliyYARf2V9m1KN9Z1Sn3afGV5BKXr5mWaEDBOFthPIjQv6LYRnGln9fRobkqMUb10Ooq3FUasI8Kyu9pGbiCQuRRQCZ1OADc'
    'J/YsuwoJ1MguvbUCVWir4T2Gtz25EodcJBc9PIy8V0eHoHGXNgoZDgr7LY8N7Y0+JhT45746FkTnIIkV1i/KcBZsuU9Am6N+NI4F'
    'qV3ldlLJ7SCuKA/h3xvadNDlcpwKdu/gF+3iHLH+ie/GQWt+kRivTj6iFej9R1ZZFim+0Ib+qEnLyRVBUgnpHTIOYqmC9PCMwaXU'
    'qTxMqJSkqVRYKnjnFPL2gqKxwdM8t6BBkFegbdT/nYFtz2qRffUJKMG08N5Nt3Q6q2L+3pKHrWc06dLU3kGGh4+51QwUVMNuudxR'
    'LaSw/6BHG38EWetaxAA1WV1cqvUGHAHxTgX/OK1CREV4CKzCi6EGp9GgI/5uYQRNqfJVZjyvQJPfpmqDrmqZ+Fl5vlJoUbyzG+mo'
    'lMeSVTs+aoBlBW7WuSg2VJmnPlr6PfyioAWyyW0S/7FJu1fB5VSG2/G+SoLipkkSkY7mMWyT2kAlTf72aqUsbvs5Z3MI35a5ysSe'
    'uKp3eGtQPvZkFp2OSqWFe92Gwf121WhkTKGUochMHY4c5/3WVm2UnXptUV2HkDoQE6RUsXDBl2m0+u9/k6hDdMb/VYI09fhXRsJp'
    'Wf+YBep7iBL+yAjbSCY82bVbhU/X+J4+T5+nz9Pn6fP0efo8fZ4+T5+nz9Pn3/zzPxn+2hQAyAAA'
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})


In [ ]:
#@title 3. Khôi phục hoặc dựng 48 khung MTCNN–PRNet
def prepare_prnet_environment():
    global PY, PRNET, RENDERER
    run("install-uv", sys.executable, "-m", "pip", "install", "uv")
    VENV = WORK / "venv"
    PY = VENV / "bin/python"
    if not PY.is_file():
        run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
    run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
        *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))
    PRNET = WORK / "PRNet"
    RENDERER = WORK / "PRNet-Depth-Generation"
    for path, repo, commit in [
        (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
        (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
    ]:
        if not path.exists():
            run(f"clone-{path.name}", "git", "clone", repo, path)
        run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
    WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
    def weight_matches(path):
        if not path.is_file():
            return False
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

    if not weight_matches(WEIGHT):
        partial = WEIGHT.with_suffix(".part")
        run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
            "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
        if not weight_matches(partial):
            raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
        partial.replace(WEIGHT)

import json
import zipfile

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

archives = {}
for name in ("Train_files.tar", "Dev_files.tar"):
    path = DATA / name
    with path.open("rb") as stream:
        archives[name] = {"bytes": path.stat().st_size,
                          "prefix_sha256": hashlib.sha256(stream.read(65536)).hexdigest()}
worker = (WORK / "scripts/cdcn_prnet_smoke.py").read_text(encoding="utf-8")
key = {"profile": "p1-3-videos-per-split-8-frames",
       "protocol": sha(DATA / "Protocols.tar"), "archives": archives,
       "preparer": hashlib.sha256(worker.encode("utf-8")).hexdigest()}
key_hash = hashlib.sha256(json.dumps(key, sort_keys=True).encode()).hexdigest()[:20]
cache = RUN.parent / "prepared-cache" / key_hash
cache.mkdir(parents=True, exist_ok=True)
assets = (WORK / f"assets-{time.time_ns()}").resolve()
cache_meta = cache / "snapshot.json"
if cache_meta.is_file():
    with Progress("Restore RGB/bbox/PRNet from Drive"):
        meta = json.loads(cache_meta.read_text())
        digest = meta["sha256"]
        if meta["key"] != key or len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest):
            raise ValueError(f"Invalid snapshot metadata: {cache_meta}")
        snapshot = cache / f"assets-{digest}.zip"
        local = assets.with_suffix(".zip")
        shutil.copyfile(snapshot, local)
        if digest != sha(local):
            raise ValueError(f"Snapshot verification failed: {snapshot}")
        with zipfile.ZipFile(local) as archive:
            if any(not (assets / name).resolve().is_relative_to(assets) for name in archive.namelist()):
                raise ValueError("Invalid archive path")
            archive.extractall(assets)
        local.unlink()
else:
    prepare_prnet_environment()
    run("prepare-48-frames", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
        "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--training-pilot", "--output", assets)
    with Progress("Save RGB/bbox/PRNet to Drive"):
        local = Path(shutil.make_archive(str(assets), "zip", assets))
        digest = sha(local)
        snapshot = cache / f"assets-{digest}.zip"
        partial = cache / f"{RUN.name}-assets.zip.part"
        shutil.copyfile(local, partial)
        if sha(partial) != digest:
            raise ValueError("Snapshot copy differs from local ZIP")
        partial.replace(snapshot)
        metadata = cache / f"{RUN.name}-snapshot.json.part"
        metadata.write_text(json.dumps({"key": key, "sha256": digest}, indent=2))
        metadata.replace(cache_meta)
        local.unlink()
shutil.copyfile(assets / "smoke_summary.json", RUN / "preparation.json")
print("PILOT_DATA_READY", snapshot)

In [ ]:
#@title 4. Train CDCN, đọc lại checkpoint và tính điểm Dev
EPOCHS = 3
BATCH = 3
SEED = 43
result = RUN / f"train-{time.time_ns()}"
run("train-and-dev", sys.executable, "-u", WORK / "scripts/cdcn_prnet_train_pilot.py",
    "--assets", assets, "--output", result, "--epochs", EPOCHS, "--batch", BATCH,
    "--seed", SEED, "--device", "cuda")
print((result / "pilot_result.json").read_text())
print("RUN_DIR", result)